# Traductor Automático - Entrenamiento del Modelo Seq2Seq

**Fase 3 (Modeling) + Fase 4 (Evaluation Preliminary)**

Entrenamiento del modelo Seq2Seq sobre datos preparados.
Monitoreo de Loss y Accuracy durante entrenamiento.
Guardado del modelo y generación de métricas.

In [ ]:
import os
import sys
import numpy as np
import json
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/honorio/IA/rnn/src')

from model import create_seq2seq_model
from preprocessing import ParallelDataProcessor

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

base_path = Path('/home/honorio/IA/rnn')
splits_path = base_path / 'data' / 'splits'
processed_path = base_path / 'data' / 'processed'
models_path = base_path / 'models'
models_path.mkdir(exist_ok=True)

print(f"✓ Setup completado")

## 1. Cargar Datos Preparados

In [ ]:
X_train = np.load(splits_path / 'X_train.npy')
y_train = np.load(splits_path / 'y_train.npy')
X_val = np.load(splits_path / 'X_val.npy')
y_val = np.load(splits_path / 'y_val.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

# Cargar vocabularios
with open(processed_path / 'vocab_spanish.json', 'r') as f:
    vocab_spanish = json.load(f)
with open(processed_path / 'vocab_english.json', 'r') as f:
    vocab_english = json.load(f)

print(f"Vocabulario español: {len(vocab_spanish)} palabras")
print(f"Vocabulario inglés: {len(vocab_english)} palabras")

## 2. Crear Modelo Seq2Seq

In [ ]:
model = create_seq2seq_model(
    source_vocab_size=len(vocab_spanish),
    target_vocab_size=len(vocab_english),
    max_length=30
)

print(model.summary())

## 3. Entrenar Modelo (CRISP-ML Fase 3)

In [ ]:
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

# Entrenar
print("Iniciando entrenamiento...")
history = model.fit(
    (X_train, y_train), y_train,
    batch_size=32,
    epochs=50,
    validation_data=((X_val, y_val), y_val),
    callbacks=[early_stop],
    verbose=1
)

print("✓ Entrenamiento completado")

## 4. Visualizar Entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss durante Entrenamiento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy durante Entrenamiento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(models_path / 'training_history.png', dpi=150)
plt.show()

print("✓ Gráficas guardadas en models/training_history.png")

## 5. Guardar Modelo Entrenado

In [ ]:
# Guardar modelo
model.save(models_path / 'traductor_v1')
print(f"✓ Modelo guardado en: {models_path / 'traductor_v1'}")

# Guardar historial de entrenamiento
history_dict = {
    'loss': history.history['loss'],
    'val_loss': history.history['val_loss'],
    'accuracy': history.history['accuracy'],
    'val_accuracy': history.history['val_accuracy'],
    'epochs': len(history.history['loss']),
    'final_train_loss': float(history.history['loss'][-1]),
    'final_val_loss': float(history.history['val_loss'][-1]),
    'final_train_acc': float(history.history['accuracy'][-1]),
    'final_val_acc': float(history.history['val_accuracy'][-1])
}

with open(models_path / 'training_metrics.json', 'w') as f:
    json.dump(history_dict, f, indent=2)

print(f"✓ Métricas guardadas en: {models_path / 'training_metrics.json'}")
print(f"\n📊 RESUMEN ENTRENAMIENTO:")
print(f"  Final Training Loss: {history_dict['final_train_loss']:.4f}")
print(f"  Final Validation Loss: {history_dict['final_val_loss']:.4f}")
print(f"  Final Training Accuracy: {history_dict['final_train_acc']:.4f}")
print(f"  Final Validation Accuracy: {history_dict['final_val_acc']:.4f}")